In [0]:
from pyspark.sql import Row

CATALOG = "power_prices"

# ---------------------------------------------------------------
# 1. dim_zone_display: tile map grid, short labels, zone centers
# ---------------------------------------------------------------
# tile_x = column (0 = west), tile_y = row (0 = north)
display_rows = [
    # zone_code,  label,    tile_x, tile_y, center_lat, center_lon
    ("NO_4",    "NO4",     4,  0, 68.5,  16.5),
    ("SE_1",    "SE1",     5,  0, 66.8,  20.5),
    ("FI",      "FI",      6,  0, 63.0,  26.0),
    ("NO_3",    "NO3",     4,  1, 63.4,  10.4),
    ("SE_2",    "SE2",     5,  1, 63.2,  16.5),
    ("NO_5",    "NO5",     3,  2, 60.8,   6.3),
    ("NO_1",    "NO1",     4,  2, 60.3,  10.8),
    ("SE_3",    "SE3",     5,  2, 59.5,  15.5),
    ("EE",      "EE",      7,  2, 58.7,  25.5),
    ("NO_2",    "NO2",     3,  3, 58.8,   7.3),
    ("SE_4",    "SE4",     5,  3, 56.0,  13.8),
    ("LV",      "LV",      7,  3, 56.9,  24.6),
    ("IE_SEM",  "IE",      0,  4, 53.4,  -7.7),
    ("DK_1",    "DK1",     3,  4, 56.2,   9.0),
    ("DK_2",    "DK2",     4,  4, 55.5,  11.9),
    ("LT",      "LT",      7,  4, 55.3,  23.9),
    ("NL",      "NL",      2,  5, 52.2,   5.5),
    ("DE_LU",   "DE-LU",   3,  5, 51.2,  10.4),
    ("PL",      "PL",      4,  5, 52.1,  19.4),
    ("BE",      "BE",      2,  6, 50.6,   4.6),
    ("CZ",      "CZ",      4,  6, 49.8,  15.5),
    ("SK",      "SK",      5,  6, 48.7,  19.6),
    ("FR",      "FR",      2,  7, 46.6,   2.4),
    ("CH",      "CH",      3,  7, 46.8,   8.2),
    ("AT",      "AT",      4,  7, 47.6,  14.1),
    ("HU",      "HU",      5,  7, 47.2,  19.4),
    ("RO",      "RO",      6,  7, 45.9,  24.9),
    ("ES",      "ES",      1,  8, 40.2,  -3.6),
    ("IT_NORD", "IT-N",    3,  8, 45.4,  10.0),
    ("SI",      "SI",      4,  8, 46.1,  14.8),
    ("HR",      "HR",      5,  8, 45.1,  15.2),
    ("RS",      "RS",      6,  8, 44.0,  20.9),
    ("PT",      "PT",      0,  9, 39.6,  -8.0),
    ("IT_SARD", "IT-SAR",  2,  9, 40.1,   9.0),
    ("IT_CNOR", "IT-CN",   3,  9, 43.4,  11.2),
    ("BG",      "BG",      6,  9, 42.7,  25.4),
    ("IT_CSUD", "IT-CS",   4, 10, 41.9,  13.6),
    ("GR",      "GR",      6, 10, 39.1,  22.0),
    ("IT_SUD",  "IT-S",    5, 11, 41.0,  16.2),
    ("IT_SICI", "IT-SIC",  4, 12, 37.5,  14.1),
    ("IT_CALA", "IT-CAL",  5, 12, 39.0,  16.4),
]

spark.createDataFrame(
    [Row(zone_code=r[0], zone_label=r[1], tile_x=r[2], tile_y=r[3], center_lat=r[4], center_lon=r[5])
     for r in display_rows]
).createOrReplaceTempView("_display")

spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOG}.gold.dim_zone_display (
  zone_code     STRING NOT NULL COMMENT 'Bidding zone, FK to dim_bidding_zone',
  zone_label    STRING NOT NULL COMMENT 'Short label for tiles and charts, e.g. DE-LU, NO1, IT-N',
  zone_name     STRING NOT NULL COMMENT 'Plain-words zone name',
  country_name  STRING NOT NULL COMMENT 'Country name',
  region        STRING NOT NULL COMMENT 'Market region grouping',
  timezone      STRING NOT NULL COMMENT 'IANA time zone',
  tile_x        INT    NOT NULL COMMENT 'Tile map column, 0 = west',
  tile_y        INT    NOT NULL COMMENT 'Tile map row, 0 = north',
  center_lat    DOUBLE NOT NULL COMMENT 'Approximate zone center latitude, for the flow map',
  center_lon    DOUBLE NOT NULL COMMENT 'Approximate zone center longitude, for the flow map'
)
COMMENT 'Display attributes per zone: tile map grid position, short label, zone center for flow lines'
""")
spark.sql(f"""
INSERT OVERWRITE {CATALOG}.gold.dim_zone_display
SELECT z.zone_code, d.zone_label, z.zone_name, z.country_name, z.region, z.timezone,
       d.tile_x, d.tile_y, d.center_lat, d.center_lon
FROM {CATALOG}.gold.dim_bidding_zone z
JOIN _display d ON z.zone_code = d.zone_code
""")

# ---------------------------------------------------------------
# 2. Fact views (hourly star schema over silver)
# ---------------------------------------------------------------
spark.sql(f"""
CREATE OR REPLACE VIEW {CATALOG}.gold.fact_price_hourly
COMMENT 'Hourly day-ahead price per zone, keyed to date and hour dimensions'
AS SELECT
  zone_code,
  timestamp_utc,
  local_date,
  CAST(date_format(local_date, 'yyyyMMdd') AS INT) AS date_key,
  local_hour,
  price_eur_mwh,
  price_eur_mwh < 0 AS is_negative,
  source_resolution
FROM {CATALOG}.silver.price
""")

spark.sql(f"""
CREATE OR REPLACE VIEW {CATALOG}.gold.fact_load_hourly
COMMENT 'Hourly actual and day-ahead forecast load per zone, with forecast error'
AS SELECT
  zone_code,
  timestamp_utc,
  MAX(local_date) AS local_date,
  CAST(date_format(MAX(local_date), 'yyyyMMdd') AS INT) AS date_key,
  MAX(local_hour) AS local_hour,
  MAX(CASE WHEN load_type = 'actual'   THEN load_mw END) AS load_actual_mw,
  MAX(CASE WHEN load_type = 'forecast' THEN load_mw END) AS load_forecast_mw,
  MAX(CASE WHEN load_type = 'actual'   THEN load_mw END)
    - MAX(CASE WHEN load_type = 'forecast' THEN load_mw END) AS forecast_error_mw
FROM {CATALOG}.silver.load
GROUP BY zone_code, timestamp_utc
""")

spark.sql(f"""
CREATE OR REPLACE VIEW {CATALOG}.gold.fact_generation_hourly
COMMENT 'Hourly generation per zone and fuel; pumped-storage pumping and other consumption shown as negative'
AS SELECT
  g.zone_code,
  g.psr_code,
  f.fuel_group,
  f.energy_class,
  g.flow_type,
  g.timestamp_utc,
  g.local_date,
  CAST(date_format(g.local_date, 'yyyyMMdd') AS INT) AS date_key,
  g.local_hour,
  CASE WHEN g.flow_type = 'consumption' THEN -g.generation_mw ELSE g.generation_mw END AS net_generation_mw
FROM {CATALOG}.silver.generation g
LEFT JOIN {CATALOG}.gold.dim_fuel_type f ON g.psr_code = f.psr_code
""")

spark.sql(f"""
CREATE OR REPLACE VIEW {CATALOG}.gold.fact_flow_hourly
COMMENT 'Hourly net physical flow per border; positive = zone_a to zone_b, negative = zone_b to zone_a'
AS SELECT
  b.border_key,
  b.zone_a,
  b.zone_b,
  f.timestamp_utc,
  SUM(CASE WHEN f.from_zone = b.zone_a THEN f.flow_mw ELSE 0 END) AS flow_a_to_b_mw,
  SUM(CASE WHEN f.from_zone = b.zone_b THEN f.flow_mw ELSE 0 END) AS flow_b_to_a_mw,
  SUM(CASE WHEN f.from_zone = b.zone_a THEN f.flow_mw ELSE -f.flow_mw END) AS net_flow_a_to_b_mw
FROM {CATALOG}.silver.flow f
JOIN {CATALOG}.gold.dim_border b
  ON (f.from_zone = b.zone_a AND f.to_zone = b.zone_b)
  OR (f.from_zone = b.zone_b AND f.to_zone = b.zone_a)
GROUP BY b.border_key, b.zone_a, b.zone_b, f.timestamp_utc
""")

# ---------------------------------------------------------------
# 3. Checks
# ---------------------------------------------------------------
d = spark.table(f"{CATALOG}.gold.dim_zone_display")
n_zones = spark.table(f"{CATALOG}.gold.dim_bidding_zone").count()
assert d.count() == n_zones == 41, f"Display rows {d.count()} vs zones {n_zones}"
assert d.select("tile_x", "tile_y").distinct().count() == 41, "Two zones share a tile"
assert d.select("zone_label").distinct().count() == 41, "Duplicate label"

for v in ["fact_price_hourly", "fact_load_hourly", "fact_generation_hourly", "fact_flow_hourly"]:
    n = spark.table(f"{CATALOG}.gold.{v}").count()
    print(f"{v:<24} view OK, {n} rows (silver empty until the token arrives)")

print(f"dim_zone_display: {d.count()} zones, unique tiles and labels | checks passed")

# Tile map preview: the grid as text
grid = {(r.tile_x, r.tile_y): r.zone_label for r in d.collect()}
max_x = max(x for x, _ in grid)
max_y = max(y for _, y in grid)
print()
for y in range(max_y + 1):
    print("".join(f"{grid.get((x, y), '.'):^8}" for x in range(max_x + 1)))